# Deployment Notebook — AutoML IDS Model (MOJO + FastAPI)

## Goal
Deploy the final AutoML-selected intrusion detection model in a reproducible way.

This notebook will:
1. Load the final selected UNSW baseline model (AutoML leader)
2. Export the model as a MOJO + genmodel jar
3. Package preprocessing + feature list + threshold metadata
4. Generate a FastAPI inference service (REST API)
5. Provide commands to run and test locally

## Why MOJO?
MOJO is H2O’s portable model format that allows deployment **without running an H2O server**.
It is lightweight and suitable for real-time inference.


In [1]:
from pathlib import Path
import json
import shutil
import h2o

# Base paths (notebook is in AutoML/notebooks)
BASE_DIR = Path.cwd().parent
OUTPUT_DIR = BASE_DIR / "outputs"

METRICS_DIR = OUTPUT_DIR / "metrics"
TEST_FRAMES_DIR = OUTPUT_DIR / "test_frames"
MODELS_DIR = OUTPUT_DIR / "models_saved"
FEATURES_DIR = OUTPUT_DIR / "features"
MODELS_ARTIFACTS_DIR = OUTPUT_DIR / "models"  # where preprocessor.joblib usually is

# Deployment folders
DEPLOY_DIR = BASE_DIR / "deployment"
ART_DIR = DEPLOY_DIR / "artifacts"
APP_DIR = DEPLOY_DIR / "app"

ART_DIR.mkdir(parents=True, exist_ok=True)
APP_DIR.mkdir(parents=True, exist_ok=True)

# Start H2O
try:
    h2o.cluster_status()
except:
    h2o.init()

print("✅ BASE_DIR:", BASE_DIR)
print("✅ DEPLOY_DIR:", DEPLOY_DIR)
print("✅ ART_DIR:", ART_DIR)
print("✅ APP_DIR:", APP_DIR)


Checking whether there is an H2O instance running at http://localhost:54321.

C:\Users\sohib\AppData\Local\Temp\ipykernel_11328\549381573.py:26: H2ODeprecationWarning: Deprecated, use ``h2o.cluster().show_status(True)``.
  h2o.cluster_status()


 connected.


H2O_cluster_uptime:,9 hours 11 mins
H2O_cluster_timezone:,Europe/London
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.9
H2O_cluster_version_age:,2 months and 20 days
H2O_cluster_name:,H2O_from_python_sohib_0caiiv
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,2.220 Gb
H2O_cluster_total_cores:,8
H2O_cluster_allowed_cores:,8
H2O_cluster_status:,"locked, healthy"


✅ BASE_DIR: C:\Users\sohib\Documents\Final Year Project\AutoML
✅ DEPLOY_DIR: C:\Users\sohib\Documents\Final Year Project\AutoML\deployment
✅ ART_DIR: C:\Users\sohib\Documents\Final Year Project\AutoML\deployment\artifacts
✅ APP_DIR: C:\Users\sohib\Documents\Final Year Project\AutoML\deployment\app


---
## Step D1 — Identify the final model for deployment

We deploy the **UNSW baseline leader model** (CBME winner / best baseline pipeline).
The model id is loaded automatically from saved evaluation metrics.


In [2]:
# Load UNSW baseline evaluation metrics (reliable)
m_unsw_base = json.load(open(METRICS_DIR / "test_metrics_bestF1.json"))
MODEL_ID = m_unsw_base["leader_model_id"]
THRESHOLD = float(m_unsw_base["best_f1_threshold"])

print("✅ Model ID:", MODEL_ID)
print("✅ Best-F1 threshold:", THRESHOLD)

# Load model from H2O cluster memory if possible
try:
    model = h2o.get_model(MODEL_ID)
    print("✅ Loaded model from H2O cluster:", model.model_id)
except Exception as e:
    print("Model not in cluster, trying disk load...", e)
    candidates = list(MODELS_DIR.rglob(f"*{MODEL_ID}*"))
    if not candidates:
        raise FileNotFoundError(f"Model {MODEL_ID} not found in {MODELS_DIR}")
    model = h2o.load_model(str(candidates[0]))
    print("✅ Loaded model from disk:", model.model_id)


✅ Model ID: GBM_1_AutoML_1_20260213_181254
✅ Best-F1 threshold: 0.875211908812319
✅ Loaded model from H2O cluster: GBM_1_AutoML_1_20260213_181254


---
## Step D2 — Export the model as MOJO

This produces:
- a `.zip` MOJO model file
- `h2o-genmodel.jar` for Java-based inference

These two files allow deployment without an H2O server.


In [3]:
# Export MOJO to deployment artifacts folder
mojo_zip_path = model.download_mojo(path=str(ART_DIR), get_genmodel_jar=True)

print("✅ MOJO ZIP saved:", mojo_zip_path)

# Confirm jar exists
jar_path = ART_DIR / "h2o-genmodel.jar"
print("✅ genmodel jar exists:", jar_path.exists(), "->", jar_path)


✅ MOJO ZIP saved: C:\Users\sohib\Documents\Final Year Project\AutoML\deployment\artifacts\GBM_1_AutoML_1_20260213_181254.zip
✅ genmodel jar exists: True -> C:\Users\sohib\Documents\Final Year Project\AutoML\deployment\artifacts\h2o-genmodel.jar


---
## Step D3 — Package preprocessing + features + threshold metadata

To ensure correct real-world inference, we deploy:
- `preprocessor.joblib` (same transformations as training)
- `selected_features.json` (same feature order as training)
- `threshold.json` (decision threshold used for predicted label)


In [4]:
# 1) Preprocessor
src_pre = MODELS_ARTIFACTS_DIR / "preprocessor.joblib"
dst_pre = ART_DIR / "preprocessor.joblib"

if not src_pre.exists():
    raise FileNotFoundError(f"preprocessor.joblib not found at: {src_pre}")

shutil.copy2(src_pre, dst_pre)
print("✅ Copied preprocessor:", dst_pre)

# 2) Selected features
src_feats = FEATURES_DIR / "selected_features.json"
dst_feats = ART_DIR / "selected_features.json"

if not src_feats.exists():
    raise FileNotFoundError(f"selected_features.json not found at: {src_feats}")

shutil.copy2(src_feats, dst_feats)
print("✅ Copied selected features:", dst_feats)

# 3) Threshold metadata
threshold_meta = {"model_id": MODEL_ID, "best_f1_threshold": THRESHOLD}

with open(ART_DIR / "threshold.json", "w") as f:
    json.dump(threshold_meta, f, indent=2)

print("✅ Saved threshold.json")


✅ Copied preprocessor: C:\Users\sohib\Documents\Final Year Project\AutoML\deployment\artifacts\preprocessor.joblib
✅ Copied selected features: C:\Users\sohib\Documents\Final Year Project\AutoML\deployment\artifacts\selected_features.json
✅ Saved threshold.json


---
## Step D4 — Generate FastAPI inference API (REST)

We generate a lightweight API that:
- accepts a JSON feature dictionary
- applies the saved preprocessor
- runs MOJO inference
- returns probability + predicted label

Endpoints:
- `/health` (status check)
- `/predict` (inference)


In [5]:
main_py = r'''
import os
import json
import joblib
import pandas as pd
import subprocess
from fastapi import FastAPI
from pydantic import BaseModel

# Paths
BASE = os.path.abspath(os.path.join(os.path.dirname(__file__), ".."))
ART = os.path.join(BASE, "artifacts")

PREPROCESSOR_PATH = os.path.join(ART, "preprocessor.joblib")
FEATURES_PATH = os.path.join(ART, "selected_features.json")
THRESH_PATH = os.path.join(ART, "threshold.json")

# Find MOJO zip
mojo_candidates = [f for f in os.listdir(ART) if f.endswith(".zip")]
if not mojo_candidates:
    raise FileNotFoundError("No MOJO .zip found in deployment/artifacts/")
MOJO_PATH = os.path.join(ART, mojo_candidates[0])

GENMODEL_JAR = os.path.join(ART, "h2o-genmodel.jar")
if not os.path.exists(GENMODEL_JAR):
    raise FileNotFoundError("h2o-genmodel.jar not found in deployment/artifacts/")

preprocessor = joblib.load(PREPROCESSOR_PATH)
selected_features = json.load(open(FEATURES_PATH))
thr = json.load(open(THRESH_PATH))["best_f1_threshold"]

app = FastAPI(title="AutoML IDS API", version="1.0")

class Record(BaseModel):
    data: dict

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/predict")
def predict(record: Record):
    df = pd.DataFrame([record.data])

    # Ensure all expected features exist (missing -> 0)
    for col in selected_features:
        if col not in df.columns:
            df[col] = 0

    df = df[selected_features]

    # Transform
    X = preprocessor.transform(df)

    # MOJO expects CSV. Write temporary input file.
    tmp_in = os.path.join(ART, "_tmp_input.csv")
    tmp_out = os.path.join(ART, "_tmp_output.csv")
    pd.DataFrame(X).to_csv(tmp_in, index=False, header=False)

    # Run Java MOJO scoring
    cmd = [
        "java", "-cp", GENMODEL_JAR, "hex.genmodel.tools.PredictCsv",
        "--mojo", MOJO_PATH,
        "--input", tmp_in,
        "--output", tmp_out,
        "--decimal"
    ]
    subprocess.run(cmd, check=True)

    out = pd.read_csv(tmp_out)

    # Probability column: usually last column from PredictCsv output
    prob_attack = float(out.iloc[0, -1])
    pred_label = int(prob_attack >= thr)

    return {"prob_attack": prob_attack, "threshold": thr, "pred_label": pred_label}
'''

(APP_DIR / "main.py").write_text(main_py, encoding="utf-8")
print("✅ Wrote:", APP_DIR / "main.py")


✅ Wrote: C:\Users\sohib\Documents\Final Year Project\AutoML\deployment\app\main.py


---
## Step D5 — Requirements and running the API

### Requirements file
We create `deployment/requirements.txt` to install API dependencies.

### Run the API
Open terminal in your project folder:

```bash
cd deployment/app
pip install -r ../requirements.txt
uvicorn main:app --reload


In [10]:



## Cell 12 — Code (Write `requirements.txt`)


reqs = """fastapi
uvicorn
pandas
numpy
joblib
scikit-learn
pydantic
"""

(DEPLOY_DIR / "requirements.txt").write_text(reqs, encoding="utf-8")
print("✅ Wrote:", DEPLOY_DIR / "requirements.txt")


✅ Wrote: C:\Users\sohib\Documents\Final Year Project\AutoML\deployment\requirements.txt


---
## Step D6 — Java requirement (MOJO)

MOJO scoring uses Java.

Check Java:
```bash
java -version


In [12]:



## Cell 14 — Code (Create a sample JSON input file)

import json

sample_payload = {
    "data": {
        # You can include only a few features; missing ones are filled with 0
        "dur": 0.1,
        "sbytes": 300,
        "dbytes": 120,
        "rate": 10,
        "sttl": 254,
        "dttl": 252,
        "sload": 5.2,
        "dload": 2.1,
        "proto": "tcp",
        "service": "http",
        "state": "FIN"
    }
}

sample_path = DEPLOY_DIR / "sample_request.json"
with open(sample_path, "w") as f:
    json.dump(sample_payload, f, indent=2)

print("✅ Saved sample request:", sample_path)


✅ Saved sample request: C:\Users\sohib\Documents\Final Year Project\AutoML\deployment\sample_request.json


In [13]:
import json
import joblib

pre = joblib.load(ART_DIR / "preprocessor.joblib")

# ColumnTransformer stores feature names in feature_names_in_ (new sklearn)
if hasattr(pre, "feature_names_in_"):
    raw_schema = list(pre.feature_names_in_)
else:
    # fallback: if your preprocessor is a Pipeline -> grab from first step if it exists
    raw_schema = None
    if hasattr(pre, "named_steps"):
        first = list(pre.named_steps.values())[0]
        if hasattr(first, "feature_names_in_"):
            raw_schema = list(first.feature_names_in_)
    if raw_schema is None:
        raise RuntimeError("Could not auto-detect raw schema from preprocessor. Tell me your raw columns list.")

out = ART_DIR / "raw_schema.json"
json.dump(raw_schema, open(out, "w"), indent=2)
print("✅ Saved raw schema:", out)
print("Raw schema size:", len(raw_schema))
print("First 10:", raw_schema[:10])


✅ Saved raw schema: C:\Users\sohib\Documents\Final Year Project\AutoML\deployment\artifacts\raw_schema.json
Raw schema size: 42
First 10: ['dur', 'proto', 'service', 'state', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl']


---
# Step D19 — Deployment Validation (API Test)

We validate the deployed API using Swagger UI (`/docs`) by sending a sample network flow record.
The API returns:
- `prob_attack`: probability of intrusion
- `threshold`: best-F1 threshold learned during evaluation
- `pred_label`: final classification based on threshold

This demonstrates the system can be used in real time for IDS-style detection.


In [14]:
import requests, json

url = "http://127.0.0.1:8000/predict"
payload = {
  "data": {
    "dur": 0.1,
    "sbytes": 300,
    "dbytes": 120,
    "rate": 10,
    "sttl": 254,
    "dttl": 252,
    "sload": 5.2,
    "dload": 2.1,
    "proto": "tcp",
    "service": "http",
    "state": "FIN"
  }
}

r = requests.post(url, json=payload)
print("Status:", r.status_code)
print(json.dumps(r.json(), indent=2))


Status: 200
{
  "prob_attack": 0.9662980693223124,
  "threshold": 0.875211908812319,
  "pred_label": 1
}


---
# UI Deployment — Streamlit Dashboard

We build a Streamlit web app that connects to the FastAPI backend.
Users can upload CSV data, select a record, and get intrusion predictions instantly.


In [15]:
from pathlib import Path

UI_DIR = DEPLOY_DIR / "ui"
UI_DIR.mkdir(parents=True, exist_ok=True)
print("✅ UI folder created:", UI_DIR)


✅ UI folder created: C:\Users\sohib\Documents\Final Year Project\AutoML\deployment\ui


In [18]:
from pathlib import Path

BASE_DIR = Path.cwd().parent
OUTPUT_DIR = BASE_DIR / "outputs"

matches = list(OUTPUT_DIR.rglob("final_feature_names.json"))

if not matches:
    raise FileNotFoundError("final_feature_names.json not found anywhere in outputs/")
    
print("Found file(s):")
for m in matches:
    print(m)


Found file(s):
C:\Users\sohib\Documents\Final Year Project\AutoML\outputs\features\final_feature_names.json


In [19]:
import shutil

src = matches[0]
dst = BASE_DIR / "deployment" / "artifacts" / "final_feature_names.json"

shutil.copy2(src, dst)

print("✅ Copied to:", dst)


✅ Copied to: C:\Users\sohib\Documents\Final Year Project\AutoML\deployment\artifacts\final_feature_names.json
